# Phase 1: Exploratory Data Analysis (EDA) & Data Cleaning 🌾📊

Welcome to the **EDA & Data Cleaning** stage of our hackathon project. In this notebook, we will:
1. Load and inspect the raw agricultural market dataset.
2. Perform data cleaning (handling missing time-series lag variables market-by-market).
3. Conduct statistical correlation analyses between crop prices, arrivals, and climate/weather variables.
4. Visualize distributions and seasonal price trends.
5. Export the sanitized data as `cleaned_dataset.csv` for machine learning training.

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="darkgrid")
print("Libraries loaded successfully!")

In [ ]:
# Cell 2: Load Raw Data
csv_path = "final_dataset_ready.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Please upload '{csv_path}' to the notebook directory.")

df_raw = pd.read_csv(csv_path)
print("=== Dataset Overview ===")
print(f"Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"Unique Markets: {df_raw['Market'].nunique()}")
print(f"Unique Districts: {df_raw['District'].nunique()}")
print(f"Time Index Range: {df_raw['time_index'].min()} to {df_raw['time_index'].max()} weeks")

df_raw.head(3)

In [ ]:
# Cell 3: Check for Missing Values
print("=== Missing Values Summary ===")
missing_counts = df_raw.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]
if missing_cols.empty:
    print("No missing values found!")
else:
    print(missing_cols)
    print(f"\nWhy are they missing? The lag features (like 'temp_mean_lag_1' or 'neigh_price_lag_1') naturally contain null values at the start of each market's timeline because no prior data exists.")

In [ ]:
# Cell 4: Time-Series Cleaning (Market-by-Market)
df_clean = df_raw.copy()

print("Applying forward-fill and backward-fill within each market group...")
for market in df_clean['Market'].unique():
    mask = df_clean['Market'] == market
    # Clean lag nulls using time order (ffill first, then bfill for the initial weeks)
    df_clean.loc[mask] = df_clean.loc[mask].ffill().bfill()

# Double check for any remaining nulls
still_null = df_clean.isnull().sum().sum()
if still_null > 0:
    # Fallback to global means if entire columns are missing (unlikely)
    df_clean = df_clean.fillna(df_clean.mean(numeric_only=True))
    
print(f"Cleaning complete! Remaining missing values: {df_clean.isnull().sum().sum()}")

In [ ]:
# Cell 5: Analyze Target Distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df_clean['Price_Rs_Per_Quintal'], bins=30, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Distribution of Price (Rs per Quintal)')
axes[0].set_xlabel('Price')

sns.histplot(df_clean['Arrivals_Metric_Tons'], bins=30, kde=True, ax=axes[1], color='orange')
axes[1].set_title('Distribution of Arrivals (Metric Tons)')
axes[1].set_xlabel('Arrivals')
axes[1].set_yscale('log') # Log scale because arrivals are highly skewed

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Climate and Price Correlations
weather_cols = ['Price_Rs_Per_Quintal', 'Rainfall', 'rain_cum_4w', 'rain_cum_12w', 'temp_mean', 'temp_range']
corr_matrix = df_clean[weather_cols].corr()

print("=== Weather & Price Correlation Matrix ===")
print(corr_matrix['Price_Rs_Per_Quintal'].sort_values(ascending=False))

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='BrBG', vmin=-1, vmax=1)
plt.title('Correlation: Weather Variables vs. Price')
plt.show()

In [ ]:
# Cell 7: Trend Visualization (Sample Market)
sample_market = "Ahmednagar APMC"
market_data = df_clean[df_clean['Market'] == sample_market]

plt.figure(figsize=(14, 6))
plt.plot(market_data['time_index'], market_data['Price_Rs_Per_Quintal'], color='teal', label='Price (Rs/Quintal)')
plt.plot(market_data['time_index'], market_data['temp_mean'] * 50, color='red', alpha=0.3, label='Mean Temp (Scaled x50)')
plt.title(f'Historical Price vs. Temperature Trends - {sample_market}')
plt.xlabel('Time Index (Weeks)')
plt.ylabel('Value')
plt.legend()
plt.show()

In [ ]:
# Cell 8: Save Cleaned Dataset
clean_csv_path = "cleaned_dataset.csv"
df_clean.to_csv(clean_csv_path, index=False)
print(f"Cleaned dataset exported to: '{clean_csv_path}'")
print(f"We are now ready for Phase 2: Feature Engineering & Model Training!")